In [138]:
import torch 
import math
from torch import nn

In [139]:
class InputEmbedding(nn.Module):
    def __init__(self,vocab_size:int ,d_model:int)->None:
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size,d_model)
        print(self.embedding.weight)
    def forward(self, x):
        return self.embedding(x)
    
    

In [140]:
batch_size = 3
vocab_size = 12
d_model = 4

# token ids to fetch embeddings
x = torch.tensor([[0,1,2],
                  [3,4,5],
                  [7,8,9]])

print(x.shape)   # (2,3)

# Creating embedding table of shape (vocab_size, d_model)
# Here -> (6,4)
input_emb = InputEmbedding(vocab_size,d_model)

# token ids shape: (batch_size, seq_len)
# (2,3)

# after embedding lookup:
# (batch_size, seq_len) --> (batch_size, seq_len, d_model)
# (2,3) --> (2,3,4)

output = input_emb(x)

print(output)
print(output.shape)

torch.Size([3, 3])
Parameter containing:
tensor([[ 0.6006,  0.3978, -0.8914,  0.7722],
        [-0.6057,  1.9506, -0.3506,  1.4229],
        [ 0.7819,  1.1940, -1.4706,  1.2111],
        [ 0.8632, -0.8858, -1.2045,  1.4164],
        [-0.5575, -0.8918,  2.0405, -0.7939],
        [-0.7728, -0.0617,  0.2824,  0.1087],
        [-0.6098, -0.7397, -1.0733, -2.8585],
        [ 0.3319, -1.5301, -0.3752,  0.3579],
        [ 2.0227, -1.4840,  0.4836, -0.7299],
        [-0.6240, -3.0103,  2.2897,  0.6303],
        [ 1.6621,  0.8583,  1.2968, -0.3924],
        [-1.0557,  0.1545, -0.8406,  1.3646]], requires_grad=True)
tensor([[[ 0.6006,  0.3978, -0.8914,  0.7722],
         [-0.6057,  1.9506, -0.3506,  1.4229],
         [ 0.7819,  1.1940, -1.4706,  1.2111]],

        [[ 0.8632, -0.8858, -1.2045,  1.4164],
         [-0.5575, -0.8918,  2.0405, -0.7939],
         [-0.7728, -0.0617,  0.2824,  0.1087]],

        [[ 0.3319, -1.5301, -0.3752,  0.3579],
         [ 2.0227, -1.4840,  0.4836, -0.7299],
      

In [141]:
class PositionalEncoding(nn.Module):

    def __init__(self, seq_len, d_model):

        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model
        # Creating empty positional encoding matrix
        # Shape: (seq_len, d_model)
        # rows    -> positions of tokens
        # columns -> embedding dimensions
        self.pe = torch.zeros((self.seq_len, self.d_model))
        # Creating position vector
        # Example for seq_len=3:
        # [0,1,2] -> [[0],[1],[2]]
        # Shape: (seq_len,1)
        # Each row represents token position
        position = torch.arange(0, seq_len).unsqueeze(1)
        # Creating frequency/divisor terms for even dimensions
        # arange(0,d_model,2) gives:
        # [0,2,4,6...]
        # Different dimensions get different frequencies:
        # smaller dimensions -> faster oscillation
        # larger dimensions  -> slower oscillation
        # Shape: (d_model/2,)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0) / d_model))
        # Broadcasting:
        # position shape  -> (seq_len,1)
        # div_term shape  -> (d_model/2,)
        # Result shape becomes:
        # (seq_len,d_model/2)
        # Fill even columns (0,2,4...) using sine
        self.pe[:, ::2] = torch.sin(position * div_term)
        # Fill odd columns (1,3,5...) using cosine
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('positional_encoding',self.pe)
        print(self.pe)
    def forward(self,x):
            return x+self.pe[:x.shape[1]]
        

In [142]:
positional_encoding = PositionalEncoding(3,4)
output=positional_encoding(output)
print(output.shape)

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998]])
torch.Size([3, 3, 4])


In [143]:

class SelfAttention(nn.Module):

    def __init__(self, d_model):

        super().__init__()

        # Number of attention heads
        self.h = 2

        # Total embedding dimension of each token
        self.d_model = d_model

        # d_model should divide equally into heads
        assert d_model % self.h == 0

        # Features handled by each head
        # Example:
        # d_model = 8, heads = 2
        # d_k = 4
        self.d_k = d_model // self.h

        # Linear layers to create:
        # Query matrix
        # Key matrix
        # Value matrix
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)

        # Final projection layer after merging heads
        self.w_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):

        # Input shape:
        # (batch_size, seq_len, d_model)

        batch_size = query.shape[0]
        seq_len = query.shape[1]

        # -----------------------------------------
        # STEP 1:
        # Create Q, K, V matrices
        # -----------------------------------------

        query = self.w_q(query)
        key = self.w_k(key)
        value = self.w_v(value)

        # Shape:
        # (batch_size, seq_len, d_model)

        # -----------------------------------------
        # STEP 2:
        # Split embeddings into multiple heads
        # -----------------------------------------

        # Before:
        # (batch_size, seq_len, d_model)

        # After:
        # (batch_size, seq_len, heads, d_k)

        query = query.reshape(
            batch_size,
            seq_len,
            self.h,
            self.d_k
        )

        key = key.reshape(
            batch_size,
            seq_len,
            self.h,
            self.d_k
        )

        value = value.reshape(
            batch_size,
            seq_len,
            self.h,
            self.d_k
        )

        # Example:
        # (2,3,8) -> (2,3,2,4)

        # -----------------------------------------
        # STEP 3:
        # Move heads dimension before seq_len
        # -----------------------------------------

        # Before:
        # (batch, seq_len, heads, d_k)

        # After:
        # (batch, heads, seq_len, d_k)

        query = query.transpose(-3, -2)
        key = key.transpose(-3, -2)
        value = value.transpose(-3, -2)

        # Example:
        # (2,3,2,4) -> (2,2,3,4)

        # -----------------------------------------
        # STEP 4:
        # Calculate attention scores
        # -----------------------------------------

        # Formula:
        # Q @ Kᵀ

        attention_score = query @ key.transpose(-2, -1)

        # Shape:
        # (batch, heads, seq_len, seq_len)

        # Example:
        # (2,2,3,3)

        # -----------------------------------------
        # STEP 5:
        # Apply mask (used in decoder)
        # -----------------------------------------

        # Future tokens get large negative value
        # so softmax makes them nearly zero

        if mask is not None:

            attention_score = attention_score.masked_fill(
                mask == 0,
                -1e9
            )

        # -----------------------------------------
        # STEP 6:
        # Scale attention scores
        # -----------------------------------------

        attention_score = (
            attention_score /
            math.sqrt(self.d_k)
        )

        # -----------------------------------------
        # STEP 7:
        # Convert scores into probabilities
        # -----------------------------------------

        attention_score = torch.softmax(
            attention_score,
            dim=-1
        )

        # -----------------------------------------
        # STEP 8:
        # Multiply attention weights with Value
        # -----------------------------------------

        attention_score = attention_score @ value

        # Shape:
        # (batch, heads, seq_len, d_k)

        # -----------------------------------------
        # STEP 9:
        # Bring seq_len dimension back
        # -----------------------------------------

        # Before:
        # (batch, heads, seq_len, d_k)

        # After:
        # (batch, seq_len, heads, d_k)

        attention_score = attention_score.transpose(
            -3,
            -2
        )

        # -----------------------------------------
        # STEP 10:
        # Merge all heads together
        # -----------------------------------------

        # Before:
        # (batch, seq_len, heads, d_k)

        # After:
        # (batch, seq_len, d_model)

        attention_score = attention_score.reshape(
            batch_size,
            seq_len,
            self.d_model
        )

        # Example:
        # (2,3,2,4) -> (2,3,8)

        # -----------------------------------------
        # STEP 11:
        # Final linear projection
        # -----------------------------------------

        attention_score = self.w_o(
            attention_score
        )

        # Final output shape:
        # (batch_size, seq_len, d_model)

        return attention_score



In [144]:
class LayerNorm(nn.Module):
    def __init__(self,eps=10**(-9)):
        super().__init__()
        
        self.eps = eps
        
    def forward(self,x,attention_score):
        self.add = x+attention_score
        self.mean = torch.mean(self.add,dim=-1).unsqueeze(2)
        self.var = torch.var((self.add-self.mean),dim=-1).unsqueeze(2)
        self.normalize =(self.add-self.mean)/torch.sqrt(self.var+self.eps)
        return self.normalize

In [145]:
class FeefForwardNetwork(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.d_model = d_model
        self.layer = nn.Sequential(
            nn.Linear(self.d_model,2048),
            nn.ReLU(),
            nn.Linear(2048,self.d_model)
        )
    def forward(self,x):
        return self.layer(x)

In [146]:

class EncoderBLock(nn.Module):

    def __init__(
        self,
        attention,
        LayerNorm,
        fnn
    ):

        super().__init__()

        self.attention = attention
        self.LayerNorm = LayerNorm
        self.fnn = fnn

    def forward(self, x):

        # -----------------------------------
        # SELF ATTENTION
        # -----------------------------------
        # Encoder uses:
        # Q = K = V = x
        # -----------------------------------

        attention_output = self.attention(
            x,
            x,
            x
        )

        # Add & Norm
        x = self.LayerNorm(
            x,
            attention_output
        )

        # Feed Forward Network
        fnn_output = self.fnn(x)

        # Add & Norm
        x = self.LayerNorm(
            x,
            fnn_output
        )

        return x

            

In [147]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        masked_attention,
        cross_attention,
        add_norm,
        ffn
    ):

        super().__init__()

        self.masked_attention = masked_attention

        self.cross_attention = cross_attention

        self.add_norm = add_norm

        self.ffn = ffn

    def forward(
        self,
        x,
        encoder_output,
        mask
    ):

        # -----------------------------------
        # MASKED SELF ATTENTION
        # -----------------------------------

        masked_output = self.masked_attention(
            x,
            x,
            x,
            mask
        )

        # Add & Norm
        x = self.add_norm(
            x,
            masked_output
        )

        # -----------------------------------
        # CROSS ATTENTION
        # -----------------------------------

        cross_output = self.cross_attention(
            x,
            encoder_output,
            encoder_output
        )

        # Add & Norm
        x = self.add_norm(
            x,
            cross_output
        )

        # -----------------------------------
        # FEED FORWARD
        # -----------------------------------

        ffn_output = self.ffn(x)

        # Add & Norm
        x = self.add_norm(
            x,
            ffn_output
        )

        return x

In [148]:

class DecoderLayer(nn.Module):

    def __init__(
        self,
        num_layers,
        d_model
    ):

        super().__init__()

        self.layers = nn.ModuleList(

            [
                DecoderBlock(

                    SelfAttention(d_model),

                    SelfAttention(d_model),

                    LayerNorm(),

                    FeefForwardNetwork(d_model)

                )

                for _ in range(num_layers)
            ]

        )

    def forward(
        self,
        x,
        encoder_output,
        mask
    ):

        for layer in self.layers:

            x = layer(
                x,
                encoder_output,
                mask
            )

        return x



In [149]:

class Transformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        seq_len,
        num_layers
    ):

        super().__init__()

        # -----------------------------------
        # INPUT TOKEN EMBEDDING
        # -----------------------------------
        # Converts token ids into vectors
        #
        # Shape:
        # (batch, seq_len)
        # ->
        # (batch, seq_len, d_model)
        # -----------------------------------

        self.embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # -----------------------------------
        # POSITIONAL ENCODING
        # -----------------------------------
        # Adds positional information
        # to embeddings
        # -----------------------------------

        self.positional_encoding = PositionalEncoding(
            seq_len,
            d_model
        )

        # -----------------------------------
        # ENCODER STACK
        # -----------------------------------

        self.encoder = EncoderLayer(
            num_layers,
            d_model
        )

        # -----------------------------------
        # DECODER STACK
        # -----------------------------------

        self.decoder = DecoderLayer(
            num_layers,
            d_model
        )

        # -----------------------------------
        # FINAL OUTPUT LAYER
        # -----------------------------------
        # Converts decoder features into
        # vocabulary probabilities
        #
        # Shape:
        # d_model -> vocab_size
        # -----------------------------------

        self.linear = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(
        self,
        encoder_input,
        decoder_input,
        mask=None
    ):

        # -----------------------------------
        # ENCODER INPUT EMBEDDING
        # -----------------------------------

        encoder_input = self.embedding(
            encoder_input
        )

        # Add positional encoding
        encoder_input = self.positional_encoding(
            encoder_input
        )

        # -----------------------------------
        # PASS THROUGH ENCODER
        # -----------------------------------

        encoder_output = self.encoder(
            encoder_input
        )

        # -----------------------------------
        # DECODER INPUT EMBEDDING
        # -----------------------------------

        decoder_input = self.embedding(
            decoder_input
        )

        # Add positional encoding
        decoder_input = self.positional_encoding(
            decoder_input
        )

        # -----------------------------------
        # PASS THROUGH DECODER
        # -----------------------------------

        decoder_output = self.decoder(
            decoder_input,
            encoder_output,
            mask
        )

        # -----------------------------------
        # FINAL LINEAR LAYER
        # -----------------------------------
        # Convert decoder output into
        # vocabulary logits
        # -----------------------------------

        output = self.linear(
            decoder_output
        )

        # -----------------------------------
        # SOFTMAX
        # -----------------------------------
        # Converts logits into probabilities
        # -----------------------------------

        output = torch.softmax(
            output,
            dim=-1
        )

        return output


In [150]:



# ============================================================
# CREATE MASK
# ============================================================

# Lower triangular matrix
# Prevents decoder from seeing future tokens

mask = torch.tril(
    torch.ones(seq_len, seq_len)
)

print(mask)

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])


In [151]:
# ============================================================
# FORWARD PASS
# ============================================================
vocab_size = 1000

d_model = 8

seq_len = 5

num_layers = 6

model = Transformer(
    vocab_size,
    d_model,
    seq_len,
    num_layers
)
output = model(
    encoder_input,
    decoder_input,
    mask
)

print(output.shape)

tensor([[ 0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  1.0000e+00],
        [ 8.4147e-01,  5.4030e-01,  9.9833e-02,  9.9500e-01,  9.9998e-03,
          9.9995e-01,  1.0000e-03,  1.0000e+00],
        [ 9.0930e-01, -4.1615e-01,  1.9867e-01,  9.8007e-01,  1.9999e-02,
          9.9980e-01,  2.0000e-03,  1.0000e+00],
        [ 1.4112e-01, -9.8999e-01,  2.9552e-01,  9.5534e-01,  2.9995e-02,
          9.9955e-01,  3.0000e-03,  1.0000e+00],
        [-7.5680e-01, -6.5364e-01,  3.8942e-01,  9.2106e-01,  3.9989e-02,
          9.9920e-01,  4.0000e-03,  9.9999e-01]])


torch.Size([2, 5, 1000])


In [152]:
predicted_token = torch.argmax(
    output,
    dim=-1
)

print(predicted_token)

tensor([[ 92, 681, 681, 867, 237],
        [681, 681, 992, 681, 237]])


In [153]:

# ============================================================
# TARGET OUTPUT
# ==============s==============================================
# Correct next words
# Shape:
# (batch_size, seq_len)
# ============================================================

target = torch.tensor([
    [6,8,2,0,0],
    [3,5,9,0,0]
])

print(target.shape)



torch.Size([2, 5])


In [154]:
output = output.reshape(
    -1,
    vocab_size
)

target = target.reshape(-1)

In [155]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [ ]:

# ============================================================
# TRAINING LOOP
# ============================================================

epochs = 1000
target = target.reshape(-1)
for epoch in range(epochs):
    # ----------------------------------------
    # FORWARD PASS
    # ----------------------------------------
    output = model(encoder_input,decoder_input,mask)
    # ----------------------------------------
    # RESHAPE OUTPUT
    # ----------------------------------------
    output = output.reshape(
        -1,
        vocab_size
    )
    # ----------------------------------------
    # CALCULATE LOSS
    # ----------------------------------------
    loss = loss_fn(
        output,
        target
    )
    # ----------------------------------------
    # REMOVE OLD GRADIENTS
    # ----------------------------------------
    optimizer.zero_grad()
    # ----------------------------------------
    # BACKPROPAGATION
    # ----------------------------------------
    loss.backward()
    # ----------------------------------------
    # UPDATE WEIGHTS
    # ----------------------------------------
    optimizer.step()
    # ----------------------------------------
    # PRINT LOSS
    # ----------------------------------------
    if epoch%100==0:
     print(f"Epoch: {epoch} | Loss: {loss.item()}")



Epoch: 0 | Loss: 6.904432773590088
Epoch: 100 | Loss: 6.9037322998046875
Epoch: 200 | Loss: 6.902929782867432
Epoch: 300 | Loss: 6.901900291442871
Epoch: 400 | Loss: 6.900547981262207
Epoch: 500 | Loss: 6.898747444152832
Epoch: 600 | Loss: 6.8963165283203125
Epoch: 700 | Loss: 6.89300537109375
Epoch: 800 | Loss: 6.8884735107421875
Epoch: 900 | Loss: 6.882283687591553
